# Data Cleaning

> "Garbage in, garbage out." A model trained on dirty data learns dirty patterns.

Data cleaning means turning a messy raw table into a **consistent, valid, ready-to-use** table. In real projects it takes the biggest share of the time, and showing a clean, **reproducible** cleaning pipeline is one of the best things you can put in a portfolio project.

This chapter uses **one messy real-world style table** (customer records) and cleans it column by column, step by step.

**Contents**
1. The cleaning workflow
2. The messy dataset
3. Step 1: Column names
4. Step 2: Text basics (`strip`, `lower`, `title`, regex)
5. Step 3: Standardizing categories (alias map and fuzzy matching)
6. Step 4: Numbers stored as text
7. Step 5: Dates in mixed formats (with a silent-bug warning)
8. Step 6: Phone and email (regex validation)
9. Step 7: Duplicates (exact and near duplicates)
10. Step 8: Outliers (detect, then decide)
11. Step 9: Missing values after cleaning
12. Step 10: Validation (rules that must hold)
13. Step 11: One reproducible cleaning function
14. Mini AI/ML example
15. Pandas 3.x notes
16. Common mistakes
17. Interview questions
18. Practice questions with solutions
19. Cheat sheet

In [59]:
import re
import difflib
import numpy as np
import pandas as pd
from pathlib import Path

print("pandas:", pd.__version__)

DATA = Path("data")
DATA.mkdir(exist_ok=True)

pandas: 3.0.6


---
# 1. The cleaning workflow

Good cleaning follows rules:

1. **Never edit the raw data.** Keep the original file untouched, work on a copy, save the result as a **new** file.
2. **Look before you clean.** Use chapter 04 tools (`info`, `describe`, `value_counts`) to find the problems.
3. **Clean column by column**, and check the result after every step.
4. **Order matters:** structure -> text -> types -> categories -> duplicates -> outliers -> missing -> validate.
   (Example: clean the text **before** looking for duplicates, because `"Karan "` and `"karan"` are the same person.)
5. **Make it reproducible:** put the final steps in one function so anyone can re-run it on new data.
6. **Validate at the end** with explicit rules, so you can prove the data is clean.

| Problem | Typical tools |
|---|---|
| Messy column names | `str.strip().str.lower().str.replace()` |
| Extra spaces, wrong case | `str.strip()`, `str.lower()`, `str.title()` |
| Inconsistent categories | `replace`, `map`, fuzzy matching |
| Numbers stored as text | `str.replace(regex)`, `pd.to_numeric(errors="coerce")` |
| Mixed date formats | `pd.to_datetime(...)` |
| Invalid phone/email | regex with `str.replace`, `str.match` |
| Duplicates | `duplicated`, `drop_duplicates` |
| Outliers | IQR, z-score, `clip` |
| Impossible values | range checks -> NaN |

---
# 2. The messy dataset

A customer table collected from different sources. Every column has a different problem. Read it carefully and try to spot them **before** we fix them.

In [60]:
raw = pd.DataFrame({
    "Customer ID": ["001", "002", "003", "004", "005", "006", "007", "008", "009", "010", "003", "011", "012", "013"],
    " Full Name ": ["  ravi KUMAR", "Asha  Verma", "KARAN singh ", "neha Sharma", "Amit Jain", "sara khan", "Dev Patel",
                    "pooja Mehta", "Isha Rao", "Rohan Das", "Karan Singh", "tara nair", "Vikram Joshi", "Meena Iyer"],
    "City": ["indore", "Bhopal ", "DELHI", "Indore", "pune", "bhopal", "New Delhi", "Indore", "Pune", "delhi",
             "Delhi", "Bhopaal", "Indore", "Pune"],
    "Phone": ["+91 98270-12345", "9827012346", "(91) 9827 012347", "98270 12348", "09827012349", "N/A", "98270-12350",
              "9827012351", "123", "98270 12353", "9827012347", "98270 12354", "9827012355", "98270-12356"],
    "Email": ["ravi@mail.com", "ASHA@MAIL.COM", "karan@mail", "neha@mail.com", "amit@@mail.com", "sara@mail.com", None,
              "pooja@mail.com", "isha@mail.com", "rohan@mail.com", "karan@mail.com", "tara@mail.com",
              "vikram@mail.com", "meena@mail.com"],
    "Age": ["21", "22", "twenty three", "22", "-1", "24", "21", "250", "23", "24", "23", "25", "26", "27"],
    "Monthly Income": ["₹30,000", "₹ 32,000", "45000", "₹31,000", "₹33,500", "₹45,000", "₹29,000", "₹1,200,000",
                       "₹38,000", "₹52,000", "45000", "₹28,000", "₹41,000", "₹36,000"],
    "Signup Date": ["2026-01-05", "12/01/2026", "15 Feb 2026", "2026/02/20", "03-03-2026", "2026-03-09", "2026-03-15",
                    "not available", "2026-04-02", "2026-04-11", "15 Feb 2026", "2026-05-01", "2026-05-09", "2026-05-20"],
    "Gender": ["M", "female", "F", "Female", "male", "f", "M", "FEMALE", "Female", "Male", "m", "F", "male", "Female"],
    "Newsletter": ["Yes", "no", "Y", "N", "yes", "1", "0", "True", "false", "Yes", "Y", "n", "yes", "No"],
})

# keep an untouched copy of the raw file on disk (rule 1)
raw.to_csv(DATA / "customers_raw.csv", index=False)
raw

,Customer ID,Full Name,City,Phone,Email,Age,Monthly Income,Signup Date,Gender,Newsletter
0,001,ravi KUMAR,indore,+91 98270-12345,ravi@mail.com,21,"₹30,000",2026-01-05,M,Yes
1,002,Asha Verma,Bhopal,9827012346,ASHA@MAIL.COM,22,"₹ 32,000",12/01/2026,female,no
2,003,KARAN singh,DELHI,(91) 9827 012347,karan@mail,twenty three,45000,15 Feb 2026,F,Y
3,004,neha Sharma,Indore,98270 12348,neha@mail.com,22,"₹31,000",2026/02/20,Female,N
4,005,Amit Jain,pune,09827012349,amit@@mail.com,-1,"₹33,500",03-03-2026,male,yes
5,006,sara khan,bhopal,N/A,sara@mail.com,24,"₹45,000",2026-03-09,f,1
6,007,Dev Patel,New Delhi,98270-12350,NaN,21,"₹29,000",2026-03-15,M,0
7,008,pooja Mehta,Indore,9827012351,pooja@mail.com,250,"₹1,200,000",not available,FEMALE,True
8,009,Isha Rao,Pune,123,isha@mail.com,23,"₹38,000",2026-04-02,Female,false
9,010,Rohan Das,delhi,98270 12353,rohan@mail.com,24,"₹52,000",2026-04-11,Male,Yes


In [61]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Customer ID     14 non-null     str  
 1    Full Name      14 non-null     str  
 2   City            14 non-null     str  
 3   Phone           14 non-null     str  
 4   Email           13 non-null     str  
 5   Age             14 non-null     str  
 6   Monthly Income  14 non-null     str  
 7   Signup Date     14 non-null     str  
 8   Gender          14 non-null     str  
 9   Newsletter      14 non-null     str  
dtypes: str(10)
memory usage: 2.2 KB


**Problems you should notice:**
- Column names have spaces, capital letters and even a leading space (`" Full Name "`)
- Names: extra spaces, mixed case
- City: `indore`, `Bhopal `, `DELHI`, `New Delhi`, `Bhopaal` (typo)
- Phone: many formats, one `N/A`, one too short (`123`)
- Email: `ASHA@MAIL.COM` (upper case), `karan@mail` (no domain), `amit@@mail.com`, one `None`
- Age: text (`twenty three`), impossible values (`-1`, `250`)
- Income: text with `₹` and commas, one huge value (`1,200,000`)
- Signup date: five different formats and `not available`
- Gender and Newsletter: same meaning written in many ways
- Customer `003` appears **twice** (Karan Singh) with slightly different data
- **Every column is text** (`str` dtype), so nothing can be calculated yet

---
# 3. Step 1: Column names

Work on a copy, and make column names clean first. Then the rest of the code is easy to write.

In [62]:
df = raw.copy()

df.columns = (
    df.columns
    .str.strip()                    # remove spaces at the ends
    .str.lower()                    # lower case
    .str.replace(r"\s+", "_", regex=True)     # spaces -> underscore
)
df = df.rename(columns={"full_name": "name"})
df.columns.tolist()

['customer_id',
 'name',
 'city',
 'phone',
 'email',
 'age',
 'monthly_income',
 'signup_date',
 'gender',
 'newsletter']

---
# 4. Step 2: Text basics

## 4.1 The essential `.str` methods (quick demo)

In [63]:
demo = pd.Series(["  ravi   KUMAR ", "ASHA verma", "Karan-Singh", None])

print(demo.str.strip().tolist())                              # remove ends
print(demo.str.lower().tolist())
print(demo.str.upper().tolist())
print(demo.str.title().tolist())
print(demo.str.replace(r"\s+", " ", regex=True).tolist())    # many spaces -> one space
print(demo.str.len().tolist())
print(demo.str.split(expand=True))                            # split into columns

['ravi   KUMAR', 'ASHA verma', 'Karan-Singh', nan]
['  ravi   kumar ', 'asha verma', 'karan-singh', nan]
['  RAVI   KUMAR ', 'ASHA VERMA', 'KARAN-SINGH', nan]
['  Ravi   Kumar ', 'Asha Verma', 'Karan-Singh', nan]
[' ravi KUMAR ', 'ASHA verma', 'Karan-Singh', nan]
[15.0, 10.0, 11.0, nan]
             0      1
0         ravi  KUMAR
1         ASHA  verma
2  Karan-Singh    NaN
3          NaN    NaN


In [64]:
# Extract part of a string with a regex group
codes = pd.Series(["SKU-00123", "SKU-00456", "bad"])
print(codes.str.extract(r"SKU-(\d+)", expand=False).tolist())

# zero-pad numbers to a fixed width
print(pd.Series(["7", "42", "123"]).str.zfill(4).tolist())

# remove everything that is NOT a digit
print(pd.Series(["Rs. 1,200", "$99.5"]).str.replace(r"\D", "", regex=True).tolist())

['00123', '00456', nan]
['0007', '0042', '0123']
['1200', '995']


Regex cheat sheet used in cleaning:

| Pattern | Meaning |
|---|---|
| `\d` / `\D` | a digit / a non-digit |
| `\s+` | one or more spaces |
| `\w` | letter, digit or underscore |
| `[^\d]` | any character that is not a digit |
| `^` / `$` | start / end of the text |
| `+`, `*`, `?` | one or more / zero or more / optional |

> In pandas 3 (and 2), `str.replace` treats the pattern as **plain text** unless you write `regex=True`.

## 4.2 Apply to `customer_id` and `name`

In [65]:
df["customer_id"] = df["customer_id"].str.strip().str.zfill(3)      # IDs stay TEXT (keeps leading zeros)

df["name"] = (
    df["name"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.title()
)
df[["customer_id", "name"]].head(6)

,customer_id,name
0,001,Ravi Kumar
1,002,Asha Verma
2,003,Karan Singh
3,004,Neha Sharma
4,005,Amit Jain
5,006,Sara Khan


---
# 5. Step 3: Standardizing categories

First **look** at the values. `value_counts()` shows the problem.

In [66]:
print(df["city"].value_counts())

city
Indore       3
Pune         2
indore       1
Bhopal       1
DELHI        1
pune         1
bhopal       1
New Delhi    1
delhi        1
Delhi        1
Bhopaal      1
Name: count, dtype: int64


In [67]:
df["city"] = df["city"].str.strip().str.title()
print(df["city"].value_counts())

city
Indore       4
Delhi        3
Pune         3
Bhopal       2
New Delhi    1
Bhopaal      1
Name: count, dtype: int64


Case is fixed, but `New Delhi` and `Bhopaal` remain. Two approaches:

**A) Rules (alias map):** you decide. Safe and clear.
**B) Fuzzy matching:** pick the closest allowed value automatically. Good for typos, but can be wrong, so always review.

Best practice: **rules first, fuzzy second, review what is left.**

In [68]:
CITIES = ["Indore", "Bhopal", "Delhi", "Pune"]            # the allowed values
ALIASES = {"New Delhi": "Delhi"}                           # rule-based fixes

def standardize_city(value):
    if pd.isna(value):
        return np.nan
    if value in ALIASES:
        return ALIASES[value]
    if value in CITIES:
        return value
    match = difflib.get_close_matches(value, CITIES, n=1, cutoff=0.8)   # fuzzy fallback
    return match[0] if match else np.nan                                 # unknown -> NaN for review

df["city"] = df["city"].map(standardize_city)
print(df["city"].value_counts())
print("not recognised:", df["city"].isna().sum())

city
Indore    4
Delhi     4
Bhopal    3
Pune      3
Name: count, dtype: int64
not recognised: 0


## Gender and newsletter: many spellings, two meanings

In [69]:
print(raw["Gender"].str.strip().str.lower().value_counts().to_dict())
print(raw["Newsletter"].str.strip().str.lower().value_counts().to_dict())

{'female': 5, 'm': 3, 'f': 3, 'male': 3}
{'yes': 4, 'no': 2, 'y': 2, 'n': 2, '1': 1, '0': 1, 'true': 1, 'false': 1}


In [70]:
# Gender: look at the first letter after lower-casing, then map
df["gender"] = df["gender"].str.strip().str.lower().str[0].map({"m": "Male", "f": "Female"})

# Newsletter: map every spelling to True/False, "boolean" keeps NaN possible
yes_no = {"yes": True, "y": True, "true": True, "1": True,
          "no": False, "n": False, "false": False, "0": False}
df["newsletter"] = df["newsletter"].str.strip().str.lower().map(yes_no).astype("boolean")

print(df["gender"].value_counts())
print(df["newsletter"].value_counts())

gender
Female    8
Male      6
Name: count, dtype: int64
newsletter
True     8
False    6
Name: count, dtype: Int64


Unmapped values become NaN with `map`. That is useful: **after mapping, check `isna().sum()`** to catch spellings you forgot.

For ML, a clean category is not just tidy: the 5 city spellings of "Delhi" would otherwise become 5 different one-hot columns (see the ML example below).

---
# 6. Step 4: Numbers stored as text

## 6.1 Age: text, impossible values

In [71]:
age = pd.to_numeric(df["age"], errors="coerce")        # "twenty three" -> NaN
print(age.tolist())

[21.0, 22.0, nan, 22.0, -1.0, 24.0, 21.0, 250.0, 23.0, 24.0, 23.0, 25.0, 26.0, 27.0]


In [72]:
# Range rule: ages outside 16-80 are impossible -> NaN
age = age.where(age.between(16, 80))
df["age"] = age
print(df["age"].tolist())

[21.0, 22.0, nan, 22.0, nan, 24.0, 21.0, nan, 23.0, 24.0, 23.0, 25.0, 26.0, 27.0]


`where(condition)` keeps values where the condition is True and puts NaN elsewhere.

**Why NaN and not "fix"?** We do not know the real age of `-1` or `250`. Inventing one would be fake data. Mark it missing (NaN) and handle it with the missing-data methods (chapter 07).

## 6.2 Income: remove symbols, then convert

In [73]:
df["income"] = pd.to_numeric(df["monthly_income"].str.replace(r"[^\d]", "", regex=True))
df = df.drop(columns=["monthly_income"])

print(df["income"].tolist())
print(df["income"].dtype)

[30000, 32000, 45000, 31000, 33500, 45000, 29000, 1200000, 38000, 52000, 45000, 28000, 41000, 36000]
int64


- `[^\d]` removes everything except digits (`₹`, commas, spaces).
- If numbers have **decimals**, keep the dot: `str.replace(r"[^\d.]", "", regex=True)`.
- Always finish with `pd.to_numeric(...)` so the column becomes a real number column.

---
# 7. Step 5: Dates in mixed formats

Our dates come in many formats: `2026-01-05`, `12/01/2026`, `15 Feb 2026`, `2026/02/20`, `03-03-2026`, `not available`.

## 7.1 The silent bug (very important)

`format="mixed"` lets pandas parse each value on its own. When you also use `dayfirst=True`, ambiguous dates such as `12/01/2026` are interpreted as day-first. The important lesson is: **mixed date formats can be ambiguous, so do not blindly trust the parsed result. Always verify a few rows against the original text.**

In [74]:
sample = df["signup_date"].head(3)
wrong = pd.to_datetime(sample, format="mixed", dayfirst=True, errors="coerce")

print(pd.DataFrame({"text": sample, "parsed (wrong!)": wrong}))

          text parsed (wrong!)
0   2026-01-05      2026-05-01
1   12/01/2026      2026-01-12
2  15 Feb 2026      2026-02-15


The exact interpretation of an ambiguous date depends on the input format and parser behavior. The dangerous part is that an incorrectly interpreted date may still look perfectly valid. That makes date parsing a potential **silent data-cleaning bug**.

## 7.2 The safe way: two passes

1. Parse the **ISO format** (`YYYY-MM-DD`) with `format="ISO8601"` (no ambiguity).
2. Parse the **remaining** values with `format="mixed", dayfirst=True`.

In [75]:
text = df["signup_date"].str.strip()

first = pd.to_datetime(text, format="ISO8601", errors="coerce")              # pass 1: ISO dates
second = pd.to_datetime(text.where(first.isna()), format="mixed",
                        dayfirst=True, errors="coerce")                       # pass 2: the rest

df["signup_date"] = first.fillna(second)
print(df[["customer_id", "signup_date"]].assign(original=raw["Signup Date"]).head(8))
print()
print("unparsed (NaT):", df["signup_date"].isna().sum())
print(df["signup_date"].dtype)

  customer_id signup_date       original
0         001  2026-01-05     2026-01-05
1         002  2026-01-12     12/01/2026
2         003  2026-02-15    15 Feb 2026
3         004  2026-02-20     2026/02/20
4         005  2026-03-03     03-03-2026
5         006  2026-03-09     2026-03-09
6         007  2026-03-15     2026-03-15
7         008         NaT  not available

unparsed (NaT): 1
datetime64[us]


Lessons:
- **Always verify** parsed dates against the original text on a few rows.
- Ambiguous formats like `03-03-2026` or `12/01/2026` need an **assumption** (day first here). Write it down in your project notes.
- `errors="coerce"` turns `not available` into `NaT` (missing date).
- If you **know the exact format**, the safest is `pd.to_datetime(s, format="%d/%m/%Y")`.

## 7.3 Use the date

In [76]:
df["signup_month"] = df["signup_date"].dt.month_name()
df["days_since_signup"] = (pd.Timestamp("2026-06-30") - df["signup_date"]).dt.days
df[["customer_id", "signup_date", "signup_month", "days_since_signup"]].head(5)

,customer_id,signup_date,signup_month,days_since_signup
0,001,2026-01-05,January,176.0
1,002,2026-01-12,January,169.0
2,003,2026-02-15,February,135.0
3,004,2026-02-20,February,130.0
4,005,2026-03-03,March,119.0


---
# 8. Step 6: Phone and email

## 8.1 Phone: keep digits, keep the last 10, validate

In [77]:
digits = df["phone"].str.replace(r"\D", "", regex=True)          # keep only digits
phone = digits.str[-10:]                                            # drop country code / leading 0

valid = (phone.str.len() == 10) & phone.str.match(r"^[6-9]")        # Indian mobile: 10 digits, starts with 6-9
df["phone"] = phone.where(valid)

print(pd.DataFrame({"original": raw["Phone"], "clean": df["phone"]}).head(10))

           original       clean
0   +91 98270-12345  9827012345
1        9827012346  9827012346
2  (91) 9827 012347  9827012347
3       98270 12348  9827012348
4       09827012349  9827012349
5               N/A         NaN
6       98270-12350  9827012350
7        9827012351  9827012351
8               123         NaN
9       98270 12353  9827012353


## 8.2 Email: lower-case, validate with a regex

In [78]:
email = df["email"].str.strip().str.lower()
valid_email = email.str.match(r"^[\w.+-]+@[\w-]+\.[\w.]+$", na=False)

df["email"] = email.where(valid_email)
print(pd.DataFrame({"original": raw["Email"], "clean": df["email"]}))

           original            clean
0     ravi@mail.com    ravi@mail.com
1     ASHA@MAIL.COM    asha@mail.com
2        karan@mail              NaN
3     neha@mail.com    neha@mail.com
4    amit@@mail.com              NaN
5     sara@mail.com    sara@mail.com
6               NaN              NaN
7    pooja@mail.com   pooja@mail.com
8     isha@mail.com    isha@mail.com
9    rohan@mail.com   rohan@mail.com
10   karan@mail.com   karan@mail.com
11    tara@mail.com    tara@mail.com
12  vikram@mail.com  vikram@mail.com
13   meena@mail.com   meena@mail.com


Regex validation checks the **shape** only. Whether an email or phone really exists is a different problem. Invalid values are set to NaN, so the missing-data tools can handle them.

---
# 9. Step 7: Duplicates

Customer `003` appears twice. Look at the **raw** rows first:

In [79]:
raw[raw["Customer ID"] == "003"]

,Customer ID,Full Name,City,Phone,Email,Age,Monthly Income,Signup Date,Gender,Newsletter
2,003,KARAN singh,DELHI,(91) 9827 012347,karan@mail,twenty three,45000,15 Feb 2026,F,Y
10,003,Karan Singh,Delhi,9827012347,karan@mail.com,23,45000,15 Feb 2026,m,Y


In [80]:
# On the raw data exact duplicates are NOT found, because the text differs a little
print("exact duplicates in raw:", raw.duplicated().sum())

# After cleaning, the records are comparable
print("duplicate customer_id after cleaning:", df.duplicated(subset=["customer_id"]).sum())
print("exact duplicates after cleaning     :", df.duplicated().sum())

exact duplicates in raw: 0
duplicate customer_id after cleaning: 1
exact duplicates after cleaning     : 0


In [81]:
df[df["customer_id"] == "003"]

,customer_id,name,city,phone,email,age,signup_date,gender,newsletter,income,signup_month,days_since_signup
2,003,Karan Singh,Delhi,9827012347,NaN,NaN,2026-02-15,Female,True,45000,February,135.0
10,003,Karan Singh,Delhi,9827012347,karan@mail.com,23.0,2026-02-15,Male,True,45000,February,135.0


Same person, but not an **exact** duplicate (the email differs: one is invalid and became NaN). This is a **near duplicate**. Rules:

- Decide the **key** that identifies a record (here `customer_id`).
- `drop_duplicates(subset=[key])` keeps one row per key.
- `keep="first"` (default), `keep="last"` or `keep=False` (drop all copies).
- To keep the **most recent** record, sort by date first, then `keep="last"`.

In [82]:
before = len(df)
df = (
    df.sort_values("signup_date")                     # oldest first
      .drop_duplicates(subset=["customer_id"], keep="last")     # keep the latest record
      .sort_values("customer_id")
      .reset_index(drop=True)
)
print(f"rows: {before} -> {len(df)}")
df[df["customer_id"] == "003"]

rows: 14 -> 13


,customer_id,name,city,phone,email,age,signup_date,gender,newsletter,income,signup_month,days_since_signup
2,003,Karan Singh,Delhi,9827012347,karan@mail.com,23.0,2026-02-15,Male,True,45000,February,135.0


---
# 10. Step 8: Outliers

An outlier is a value far from the rest. First **detect**, then **decide** (never delete blindly).

## 10.1 Detect with the IQR rule and the z-score

In [83]:
inc = df["income"]

q1, q3 = inc.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"IQR limits: {low:.0f} to {high:.0f}")

z = (inc - inc.mean()) / inc.std()
print("z-score outliers (|z| > 3):", (z.abs() > 3).sum())
print("IQR outliers              :", ((inc < low) | (inc > high)).sum())

df[(inc < low) | (inc > high)][["customer_id", "name", "income"]]

IQR limits: 10000 to 66000
z-score outliers (|z| > 3): 1
IQR outliers              : 1


,customer_id,name,income
7,008,Pooja Mehta,1200000


In [84]:
print("z-score of the huge income:", round(z.max(), 2), "(barely above 3 although it is 25x a normal income)")

z-score of the huge income: 3.33 (barely above 3 although it is 25x a normal income)


Both methods catch it here, but look at the z-score: only about 3.3. The huge value inflates the mean and the std, so it **hides itself**. This is called **masking**. Watch what happens with **two** extreme values:

In [85]:
inc2 = df["income"].copy()
inc2.iloc[0] = 900_000                              # a second extreme value

z2 = (inc2 - inc2.mean()) / inc2.std()
q1b, q3b = inc2.quantile([0.25, 0.75])
iqr_b = q3b - q1b

print("z-score finds :", (z2.abs() > 3).sum(), "outliers")
print("IQR finds     :", ((inc2 < q1b - 1.5 * iqr_b) | (inc2 > q3b + 1.5 * iqr_b)).sum(), "outliers")

z-score finds : 0 outliers
IQR finds     : 2 outliers


The z-score finds **none** (the two big values inflate the std so much that neither looks extreme), while IQR still finds both, because quartiles are not affected by extreme values. For small or skewed data, prefer **IQR**.

## 10.2 Decide what to do

| Option | When | How |
|---|---|---|
| **Fix** | It is a typo (extra zeros) | Correct it using domain knowledge |
| **Remove** | Proven error, tiny share of data | `df[~mask]` |
| **Cap (winsorize)** | Real but extreme, and it hurts the model | `clip(lower, upper)` |
| **Transform** | Strong right skew (income, price) | `np.log1p` |
| **Keep + flag** | Real and important (fraud, rare event) | add `is_outlier` column |

Good habit: **keep the original column, add new ones**, so decisions are reversible.

In [86]:
df["income_outlier"] = (df["income"] < low) | (df["income"] > high)     # flag
df["income_capped"] = df["income"].clip(lower=low, upper=high)          # capped copy
df["income_log"] = np.log1p(df["income"])                                # log transform

df.loc[df["income_outlier"], ["name", "income", "income_capped", "income_log"]].round(2)

,name,income,income_capped,income_log
7,Pooja Mehta,1200000,66000,14.0


---
# 11. Step 9: Missing values after cleaning

Cleaning turns invalid values into NaN. Now see the overall picture (filling methods were covered in chapter 07).

In [87]:
report = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(1),
})
report[report["missing"] > 0]

,missing,missing_%
phone,2,15.4
email,2,15.4
age,2,15.4
signup_date,1,7.7
signup_month,1,7.7
days_since_signup,1,7.7


Remember the chapter 07 decision rules: age and email can be filled or left missing (add an indicator if useful); a missing `signup_date` may be better left as NaT than invented.

---
# 12. Step 10: Validation

After cleaning, **prove** the data follows the rules. Write the rules as code, so they can run again every time new data arrives.

In [88]:
def validate(d, reference_date=pd.Timestamp("2026-06-30")):
    rules = {
        "customer_id is unique":              d["customer_id"].is_unique,
        "age between 16 and 80 (or missing)": d["age"].dropna().between(16, 80).all(),
        "phone has 10 digits (or missing)":   d["phone"].dropna().str.len().eq(10).all(),
        "email looks valid (or missing)":     d["email"].dropna().str.match(
            r"^[\w.+-]+@[\w-]+\.[\w.]+$"
        ).all(),
        "city in allowed list":               d["city"].isin(CITIES).all(),
        "gender is Male/Female (or missing)": d["gender"].dropna().isin(["Male", "Female"]).all(),
        "income is positive":                 (d["income"] > 0).all(),
        "no signup date in the future":       (
            d["signup_date"].dropna() <= reference_date
        ).all(),
        "names have no extra spaces":         (
            d["name"] ==
            d["name"].str.strip().str.replace(r"\s+", " ", regex=True)
        ).all(),
    }

    return pd.Series(rules, name="passed")

validate(df, reference_date=pd.Timestamp("2026-10-06"))

customer_id is unique                 True
age between 16 and 80 (or missing)    True
phone has 10 digits (or missing)      True
email looks valid (or missing)        True
city in allowed list                  True
gender is Male/Female (or missing)    True
income is positive                    True
no signup date in the future          True
names have no extra spaces            True
Name: passed, dtype: bool

In [89]:
# Prove the validator catches problems: break the data on purpose
bad = df.copy()
bad.loc[0, "age"] = 250
bad.loc[1, "city"] = "Mumbai"
bad.loc[2, "phone"] = "123"

result = validate(bad)
print(result[~result])

age between 16 and 80 (or missing)    False
phone has 10 digits (or missing)      False
city in allowed list                  False
Name: passed, dtype: bool


In [90]:
# In a pipeline you can stop the run when a rule fails:
assert validate(df).all(), "Validation failed!"
print("All validation rules passed")

All validation rules passed


---
# 13. Step 11: One reproducible cleaning function

Everything we did, in one function: **raw in, clean out**. This is what you commit to GitHub in a real project. New data arrives next month? Run the same function.

In [91]:
CITIES = ["Indore", "Bhopal", "Delhi", "Pune"]
ALIASES = {"New Delhi": "Delhi"}
YES_NO = {"yes": True, "y": True, "true": True, "1": True, "no": False, "n": False, "false": False, "0": False}
EMAIL_RE = r"^[\w.+-]+@[\w-]+\.[\w.]+$"


def clean_customers(raw_df):
    d = raw_df.copy()

    # column names
    d.columns = d.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)
    d = d.rename(columns={"full_name": "name"})

    # text
    d["customer_id"] = d["customer_id"].str.strip().str.zfill(3)
    d["name"] = d["name"].str.strip().str.replace(r"\s+", " ", regex=True).str.title()

    # categories
    d["city"] = d["city"].str.strip().str.title().map(standardize_city)
    d["gender"] = d["gender"].str.strip().str.lower().str[0].map({"m": "Male", "f": "Female"})
    d["newsletter"] = d["newsletter"].str.strip().str.lower().map(YES_NO).astype("boolean")

    # numbers
    age = pd.to_numeric(d["age"], errors="coerce")
    d["age"] = age.where(age.between(16, 80))
    d["income"] = pd.to_numeric(d["monthly_income"].str.replace(r"[^\d]", "", regex=True))
    d = d.drop(columns=["monthly_income"])

    # dates (two passes)
    text = d["signup_date"].str.strip()
    first = pd.to_datetime(text, format="ISO8601", errors="coerce")
    second = pd.to_datetime(text.where(first.isna()), format="mixed", dayfirst=True, errors="coerce")
    d["signup_date"] = first.fillna(second)

    # phone and email
    phone = d["phone"].str.replace(r"\D", "", regex=True).str[-10:]
    d["phone"] = phone.where((phone.str.len() == 10) & phone.str.match(r"^[6-9]"))
    email = d["email"].str.strip().str.lower()
    d["email"] = email.where(email.str.match(EMAIL_RE, na=False))

    # duplicates: keep the latest record per customer
    d = (d.sort_values("signup_date")
          .drop_duplicates(subset=["customer_id"], keep="last")
          .sort_values("customer_id")
          .reset_index(drop=True))

    # outliers: flag and cap (original column is kept)
    q1, q3 = d["income"].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    d["income_outlier"] = (d["income"] < low) | (d["income"] > high)
    d["income_capped"] = d["income"].clip(lower=low, upper=high)

    return d


clean = clean_customers(raw)
assert validate(clean).all()
print(clean.shape)
clean

(13, 12)


,customer_id,name,city,phone,email,age,signup_date,gender,newsletter,income,income_outlier,income_capped
0,001,Ravi Kumar,Indore,9827012345,ravi@mail.com,21.0,2026-01-05,Male,True,30000,False,30000
1,002,Asha Verma,Bhopal,9827012346,asha@mail.com,22.0,2026-01-12,Female,False,32000,False,32000
2,003,Karan Singh,Delhi,9827012347,karan@mail.com,23.0,2026-02-15,Male,True,45000,False,45000
3,004,Neha Sharma,Indore,9827012348,neha@mail.com,22.0,2026-02-20,Female,False,31000,False,31000
4,005,Amit Jain,Pune,9827012349,NaN,NaN,2026-03-03,Male,True,33500,False,33500
5,006,Sara Khan,Bhopal,NaN,sara@mail.com,24.0,2026-03-09,Female,True,45000,False,45000
6,007,Dev Patel,Delhi,9827012350,NaN,21.0,2026-03-15,Male,False,29000,False,29000
7,008,Pooja Mehta,Indore,9827012351,pooja@mail.com,NaN,NaT,Female,True,1200000,True,66000
8,009,Isha Rao,Pune,NaN,isha@mail.com,23.0,2026-04-02,Female,False,38000,False,38000
9,010,Rohan Das,Delhi,9827012353,rohan@mail.com,24.0,2026-04-11,Male,True,52000,False,52000


In [92]:
# The function must give the same result as our step-by-step work
same_cols = ["customer_id", "name", "city", "phone", "email", "age", "income", "gender", "newsletter",
             "signup_date", "income_outlier", "income_capped"]
pd.testing.assert_frame_equal(clean[same_cols], df[same_cols], check_dtype=False)
print("function result == step-by-step result")

function result == step-by-step result


In [93]:
# Before / after summary (great for a project report)
summary = pd.DataFrame({
    "before": {
        "rows": len(raw),
        "unique city spellings": raw["City"].nunique(),
        "numeric columns": raw.select_dtypes(include="number").shape[1],
        "duplicate customers": raw["Customer ID"].duplicated().sum(),
    },
    "after": {
        "rows": len(clean),
        "unique city spellings": clean["city"].nunique(),
        "numeric columns": clean.select_dtypes(include="number").shape[1],
        "duplicate customers": clean["customer_id"].duplicated().sum(),
    },
})
summary

,before,after
rows,14,13
unique city spellings,11,4
numeric columns,0,3
duplicate customers,1,0


In [94]:
# Save the CLEAN data as a separate file. The raw file stays untouched.
clean.to_csv(DATA / "customers_clean.csv", index=False)
print(sorted(p.name for p in DATA.glob("customers_*")))

['customers_clean.csv', 'customers_raw.csv']


---
# 14. Mini AI/ML example

Why does cleaning matter for a model? Look at the **number of features** created by one-hot encoding, before and after.

In [95]:
before_cols = pd.get_dummies(raw["City"]).shape[1]
after_cols = pd.get_dummies(clean["city"]).shape[1]

print("one-hot columns for city from RAW  :", before_cols)
print("one-hot columns for city from CLEAN:", after_cols)

one-hot columns for city from RAW  : 11
one-hot columns for city from CLEAN: 4


The same 4 cities became **11 different features** in the raw data. The model would waste capacity, learn from tiny groups and fail on new spellings. Clean categories give clean features.

### Outlier limits must come from the training set only

Just like missing-value fills (chapter 07), outlier limits are **learned values**. Learn them from the **train** part and apply the same limits to the test part. Otherwise test data leaks into training.

In [96]:
train, test = clean.iloc[:10].copy(), clean.iloc[10:].copy()

q1, q3 = train["income"].quantile([0.25, 0.75])
iqr = q3 - q1
low_t, high_t = q1 - 1.5 * iqr, q3 + 1.5 * iqr          # learned from TRAIN only
print(f"limits from train: {low_t:.0f} to {high_t:.0f}")

train["income_capped"] = train["income"].clip(low_t, high_t)
test["income_capped"] = test["income"].clip(low_t, high_t)    # same limits applied to test

limits from train: 10625 to 65625


In [97]:
# Final model-ready TRAIN table
train_features = pd.get_dummies(
    train[["age", "income_capped", "city", "gender"]],
    columns=["city", "gender"],
    dtype=int
)

# Final model-ready TEST table
test_features = pd.get_dummies(
    test[["age", "income_capped", "city", "gender"]],
    columns=["city", "gender"],
    dtype=int
)

# Make train and test columns identical
test_features = test_features.reindex(
    columns=train_features.columns,
    fill_value=0
)

# Fill age using TRAIN median only
train_age_median = train_features["age"].median()

train_features["age"] = train_features["age"].fillna(train_age_median)
test_features["age"] = test_features["age"].fillna(train_age_median)

print("train features shape:", train_features.shape)
print("test features shape :", test_features.shape)

train_features.head(4)

train features shape: (10, 8)
test features shape : (3, 8)


,age,income_capped,city_Bhopal,city_Delhi,city_Indore,city_Pune,gender_Female,gender_Male
0,21.0,30000,0,0,1,0,0,1
1,22.0,32000,1,0,0,0,1,0
2,23.0,45000,0,1,0,0,0,1
3,22.0,31000,0,0,1,0,1,0


---
# 15. Pandas 3.x notes

- Text columns are `str` dtype. `.str` methods work as before, and missing text stays `NaN`.
- `str.replace(pattern, repl)` needs **`regex=True`** to use regex.
- `pd.to_datetime(..., format="mixed")` is the way to parse mixed formats. Combined with `dayfirst=True` it can silently misread ISO dates, so use the two-pass approach.
- `astype("boolean")` (nullable) keeps missing values. Plain `bool` cannot hold NaN.
- Always assign results back (`df["a"] = df["a"].str.strip()`). Cleaning methods return new objects.

---
# 16. Common mistakes

| Mistake | Fix |
|---|---|
| Editing the raw file/DataFrame | Work on a copy, save clean data as a new file |
| Looking for duplicates **before** cleaning text | Clean first (`strip`, `lower`), then `drop_duplicates` |
| `str.replace(r"\D", "")` without `regex=True` | `str.replace(r"\D", "", regex=True)` |
| Removing `.` from numbers with decimals | Use `[^\d.]` |
| Converting IDs/phones/pincodes to numbers | Keep them as text (leading zeros) |
| `dayfirst=True` on ISO dates with `format="mixed"` | Two-pass parsing, and verify on a few rows |
| Trusting the parsed dates without checking | Compare with the original text |
| Filling impossible values (age 250) with guesses | Set to NaN, then use proper missing-value handling |
| Deleting every outlier | Decide: fix, cap, transform or flag |
| Using z-score on small/skewed data | Use IQR |
| Fuzzy matching with no review | Use rules first, check what fuzzy matched |
| Cleaning by hand in many scattered cells | One reproducible function + validation rules |
| Computing outlier limits on all data before the split | Learn limits from train only |
| No validation at the end | Write rules and `assert` them |

---
# 17. Interview questions

**Q1. What are the main steps of data cleaning?**
Inspect the data, fix column names, clean text, fix types, standardize categories, handle duplicates, handle outliers, handle missing values, and validate. Keep the raw data untouched.

**Q2. How do you remove duplicates in pandas?**
`df.duplicated()` to find them and `df.drop_duplicates(subset=[...], keep=...)` to remove. Clean the text first so near duplicates become exact. To keep the latest record, sort by date and use `keep="last"`.

**Q3. How do you clean a column like "Rs. 1,200" into a number?**
`pd.to_numeric(s.str.replace(r"[^\d.]", "", regex=True), errors="coerce")`.

**Q4. How do you handle inconsistent categories ("delhi", "Delhi ", "New Delhi")?**
Strip and normalize case, then map with an alias dictionary, and optionally use fuzzy matching (`difflib`) for typos. Review unmatched values.

**Q5. How do you detect outliers and what do you do with them?**
IQR rule (`Q1 - 1.5*IQR`, `Q3 + 1.5*IQR`), z-score, plus business rules. Then decide: fix, remove, cap with `clip`, transform with log, or just flag.

**Q6. IQR vs z-score?**
IQR uses quartiles and is robust to outliers and skewed data. Z-score uses mean and std, which are themselves distorted by outliers.

**Q7. What does `errors="coerce"` do?**
Invalid values become NaN (or NaT for dates) instead of raising an error, so bad entries can be handled afterwards.

**Q8. How do you parse dates in many different formats?**
`pd.to_datetime(..., format="mixed")`, ideally in two passes (ISO first, then `dayfirst` for the rest), and verify the result. If the format is known, give the exact `format=`.

**Q9. Why keep IDs and pincodes as strings?**
They are labels, not quantities. Converting to numbers removes leading zeros and invites meaningless maths.

**Q10. How do you make cleaning reproducible?**
Put all steps into a function (or `pipe` chain), keep raw data separate, add validation rules with `assert`, and save the clean output as a new file.

**Q11. What is data leakage in cleaning?**
Using statistics from the whole dataset (means, outlier limits, category lists) before splitting. Learn them from the train set only, then apply to test.

---
# 18. Practice questions

Run this cell first to get the messy inventory table.

In [98]:
inv = pd.DataFrame({
    "Product Name": [" laptop pro", "Mouse  ", "KEYBOARD", "monitor 24", "laptop pro", "usb cable", "Webcam", "headphones", "Mouse", "speaker"],
    "Category":     ["Electronics", "elec", "ELECTRONICS ", "Electronic", "Electronics", "accessories", "Accessory", "accessories", "elec", "Audio"],
    "Price":        ["Rs. 65,000", "Rs. 500", "₹1,200", "9500", "Rs. 64,000", "150", "Rs. 2,500", "Rs. 1,800", "Rs.550", "Rs. 99,99,999"],
    "Stock":        [12, 150, -5, 30, 12, 400, 25, 60, 140, 8],
    "Updated":      ["2026-03-01", "01/04/2026", "5 May 2026", "2026-06-10", "2026-04-01", "2026/05/20", "never", "2026-06-15", "2026-06-20", "2026-07-01"],
    "SKU":          ["SKU-00123", "SKU-00456", "SKU-00789", "SKU-01011", "SKU-00123", "SKU-01213", "SKU-01415", "SKU-01617", "SKU-00456", "SKU-01819"],
    "Brand - Model": ["Dell - XPS", "Logitech - M90", "Keychron - K2", "LG - 24MP", "Dell - XPS", "Belkin - C1", "Logi - C920", "Sony - H100", "Logitech - M90", "JBL - Go3"],
})
inv

,Product Name,Category,Price,Stock,Updated,SKU,Brand - Model
0,laptop pro,Electronics,"Rs. 65,000",12,2026-03-01,SKU-00123,Dell - XPS
1,Mouse,elec,Rs. 500,150,01/04/2026,SKU-00456,Logitech - M90
2,KEYBOARD,ELECTRONICS,"₹1,200",-5,5 May 2026,SKU-00789,Keychron - K2
3,monitor 24,Electronic,9500,30,2026-06-10,SKU-01011,LG - 24MP
4,laptop pro,Electronics,"Rs. 64,000",12,2026-04-01,SKU-00123,Dell - XPS
5,usb cable,accessories,150,400,2026/05/20,SKU-01213,Belkin - C1
6,Webcam,Accessory,"Rs. 2,500",25,never,SKU-01415,Logi - C920
7,headphones,accessories,"Rs. 1,800",60,2026-06-15,SKU-01617,Sony - H100
8,Mouse,elec,Rs.550,140,2026-06-20,SKU-00456,Logitech - M90
9,speaker,Audio,"Rs. 99,99,999",8,2026-07-01,SKU-01819,JBL - Go3


**Q1.** Convert all column names to `snake_case` (`product_name`, `category`, ..., `brand_-_model` should become `brand_model`).
**Q2.** Clean `product_name`: strip, collapse spaces, title case.
**Q3.** Convert `price` to a number (remove `Rs.`, `₹` and commas).
**Q4.** Standardize `category` into `Electronics`, `Accessories`, `Audio` (use strip/title, then an alias dictionary: `Elec`, `Electronic` -> `Electronics`; `Accessory` -> `Accessories`).
**Q5.** Convert `updated` into real dates (formats are ISO, `dd/mm/yyyy`, `d Mon yyyy`, `yyyy/mm/dd`, plus `never`). Use the two-pass method.
**Q6.** Set negative `stock` values to NaN.
**Q7.** Remove duplicate products (same cleaned `product_name`), keeping the **most recently updated** row.
**Q8.** Create an `is_price_outlier` column using the IQR rule on `price`.
**Q9.** Extract the numeric part of `sku` as an integer column `sku_number`.
**Q10.** Split `brand_model` into two columns `brand` and `model`.
**Q11.** Write a `validate_inventory(df)` function that checks: price > 0, stock >= 0 (or NaN), product names unique, category in the allowed list.
**Q12.** Put Q1 to Q10 into one function `clean_inventory(raw_df)` and run it with `pipe`.

### Solutions

In [99]:
# Q1
q1 = inv.copy()
q1.columns = (q1.columns.str.strip().str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True)      # any run of symbols/spaces -> _
              .str.strip("_"))
q1.columns.tolist()

['product_name', 'category', 'price', 'stock', 'updated', 'sku', 'brand_model']

In [100]:
# Q2 (continue using the cleaned names)
inv2 = q1.copy()
inv2["product_name"] = inv2["product_name"].str.strip().str.replace(r"\s+", " ", regex=True).str.title()
inv2["product_name"].tolist()

['Laptop Pro',
 'Mouse',
 'Keyboard',
 'Monitor 24',
 'Laptop Pro',
 'Usb Cable',
 'Webcam',
 'Headphones',
 'Mouse',
 'Speaker']

In [101]:
# Q3
inv2["price"] = pd.to_numeric(inv2["price"].str.replace(r"[^\d]", "", regex=True))
inv2["price"].tolist()

[65000, 500, 1200, 9500, 64000, 150, 2500, 1800, 550, 9999999]

In [102]:
# Q4
alias = {"Elec": "Electronics", "Electronic": "Electronics", "Accessory": "Accessories"}
inv2["category"] = inv2["category"].str.strip().str.title().replace(alias)
inv2["category"].value_counts()

category
Electronics    6
Accessories    3
Audio          1
Name: count, dtype: int64

In [103]:
# Q5
text = inv2["updated"].str.strip()
first = pd.to_datetime(text, format="ISO8601", errors="coerce")
second = pd.to_datetime(text.where(first.isna()), format="mixed", dayfirst=True, errors="coerce")
inv2["updated"] = first.fillna(second)
inv2[["updated"]].assign(original=inv["Updated"])

,updated,original
0,2026-03-01,2026-03-01
1,2026-04-01,01/04/2026
2,2026-05-05,5 May 2026
3,2026-06-10,2026-06-10
4,2026-04-01,2026-04-01
5,2026-05-20,2026/05/20
6,NaT,never
7,2026-06-15,2026-06-15
8,2026-06-20,2026-06-20
9,2026-07-01,2026-07-01


In [104]:
# Q6
inv2["stock"] = inv2["stock"].where(inv2["stock"] >= 0)
inv2["stock"].tolist()

[12.0, 150.0, nan, 30.0, 12.0, 400.0, 25.0, 60.0, 140.0, 8.0]

In [105]:
# Q7
before = len(inv2)
inv3 = (inv2.sort_values("updated")
            .drop_duplicates(subset=["product_name"], keep="last")
            .sort_index())
print(before, "->", len(inv3))
inv3[["product_name", "updated"]]

10 -> 8


,product_name,updated
2,Keyboard,2026-05-05
3,Monitor 24,2026-06-10
4,Laptop Pro,2026-04-01
5,Usb Cable,2026-05-20
6,Webcam,NaT
7,Headphones,2026-06-15
8,Mouse,2026-06-20
9,Speaker,2026-07-01


In [106]:
# Q8
# Note: IQR also flags the real 64,000 laptop, not only the typo-like 99,99,999 speaker.
# The rule only says "far from the others". Domain knowledge decides: laptop = real, speaker = error.
q1_, q3_ = inv3["price"].quantile([0.25, 0.75])
iqr = q3_ - q1_
inv3 = inv3.assign(is_price_outlier=(inv3["price"] < q1_ - 1.5 * iqr) | (inv3["price"] > q3_ + 1.5 * iqr))
inv3[["product_name", "price", "is_price_outlier"]]

,product_name,price,is_price_outlier
2,Keyboard,1200,False
3,Monitor 24,9500,False
4,Laptop Pro,64000,True
5,Usb Cable,150,False
6,Webcam,2500,False
7,Headphones,1800,False
8,Mouse,550,False
9,Speaker,9999999,True


In [107]:
# Q9
inv3["sku_number"] = inv3["sku"].str.extract(r"(\d+)", expand=False).astype(int)
inv3[["sku", "sku_number"]].head(4)

,sku,sku_number
2,SKU-00789,789
3,SKU-01011,1011
4,SKU-00123,123
5,SKU-01213,1213


In [108]:
# Q10
inv3[["brand", "model"]] = inv3["brand_model"].str.split(r"\s*-\s*", n=1, expand=True)
inv3[["brand_model", "brand", "model"]].head(4)

,brand_model,brand,model
2,Keychron - K2,Keychron,K2
3,LG - 24MP,LG,24MP
4,Dell - XPS,Dell,XPS
5,Belkin - C1,Belkin,C1


In [109]:
# Q11
ALLOWED = ["Electronics", "Accessories", "Audio"]

def validate_inventory(d):
    return pd.Series({
        "price > 0":               (d["price"] > 0).all(),
        "stock >= 0 (or NaN)":     d["stock"].dropna().ge(0).all(),
        "product names unique":    d["product_name"].is_unique,
        "category in allowed list": d["category"].isin(ALLOWED).all(),
    }, name="passed")

validate_inventory(inv3)

price > 0                   True
stock >= 0 (or NaN)         True
product names unique        True
category in allowed list    True
Name: passed, dtype: bool

In [110]:
# Q12
def clean_inventory(raw_df):
    d = raw_df.copy()
    d.columns = (d.columns.str.strip().str.lower()
                 .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))
    d["product_name"] = d["product_name"].str.strip().str.replace(r"\s+", " ", regex=True).str.title()
    d["price"] = pd.to_numeric(d["price"].str.replace(r"[^\d]", "", regex=True))
    d["category"] = d["category"].str.strip().str.title().replace(alias)
    text = d["updated"].str.strip()
    first = pd.to_datetime(text, format="ISO8601", errors="coerce")
    second = pd.to_datetime(text.where(first.isna()), format="mixed", dayfirst=True, errors="coerce")
    d["updated"] = first.fillna(second)
    d["stock"] = d["stock"].where(d["stock"] >= 0)
    d = (d.sort_values("updated").drop_duplicates(subset=["product_name"], keep="last").sort_index())
    q1_, q3_ = d["price"].quantile([0.25, 0.75])
    iqr = q3_ - q1_
    d["is_price_outlier"] = (d["price"] < q1_ - 1.5 * iqr) | (d["price"] > q3_ + 1.5 * iqr)
    d["sku_number"] = d["sku"].str.extract(r"(\d+)", expand=False).astype(int)
    d[["brand", "model"]] = d["brand_model"].str.split(r"\s*-\s*", n=1, expand=True)
    return d.drop(columns=["brand_model"]).reset_index(drop=True)

final = inv.pipe(clean_inventory)
print(validate_inventory(final).all())
final

True


,product_name,category,price,stock,updated,sku,is_price_outlier,sku_number,brand,model
0,Keyboard,Electronics,1200,NaN,2026-05-05,SKU-00789,False,789,Keychron,K2
1,Monitor 24,Electronics,9500,30.0,2026-06-10,SKU-01011,False,1011,LG,24MP
2,Laptop Pro,Electronics,64000,12.0,2026-04-01,SKU-00123,True,123,Dell,XPS
3,Usb Cable,Accessories,150,400.0,2026-05-20,SKU-01213,False,1213,Belkin,C1
4,Webcam,Accessories,2500,25.0,NaT,SKU-01415,False,1415,Logi,C920
5,Headphones,Accessories,1800,60.0,2026-06-15,SKU-01617,False,1617,Sony,H100
6,Mouse,Electronics,550,140.0,2026-06-20,SKU-00456,False,456,Logitech,M90
7,Speaker,Audio,9999999,8.0,2026-07-01,SKU-01819,True,1819,JBL,Go3


---
# 19. Cheat sheet

```python
# NAMES
df.columns = df.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)

# TEXT
s.str.strip()   s.str.lower()   s.str.title()
s.str.replace(r"\s+", " ", regex=True)           # collapse spaces
s.str.replace(r"[^\d]", "", regex=True)          # keep digits only
s.str.extract(r"(\d+)", expand=False)            # pull out a part
s.str.split(" - ", n=1, expand=True)             # split into columns
s.str.zfill(3)    s.str.match(regex)    s.str.contains("x", na=False)

# CATEGORIES
s.replace({"old": "new"})    s.map(dict)    difflib.get_close_matches(x, allowed, cutoff=0.8)

# NUMBERS and DATES
pd.to_numeric(s, errors="coerce")        s.where(s.between(a, b))     # invalid -> NaN
iso = pd.to_datetime(s, format="ISO8601", errors="coerce")
rest = pd.to_datetime(s.where(iso.isna()), format="mixed", dayfirst=True, errors="coerce")
dates = iso.fillna(rest)

# DUPLICATES
df.duplicated(subset=["id"]).sum()
df.sort_values("date").drop_duplicates(subset=["id"], keep="last")

# OUTLIERS (IQR)
q1, q3 = s.quantile([0.25, 0.75]);  iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
s.clip(low, high)       np.log1p(s)       (s < low) | (s > high)

# VALIDATE and SAVE
assert validate(df).all()
df.to_csv("data/clean.csv", index=False)         # raw stays untouched

# PIPELINE
clean = raw.pipe(clean_function)
```

**Next chapter:** `09_sorting_ranking_apply`: `sort_values`, `sort_index`, `rank`, `nlargest`, `apply`, `map`, `applymap`/`DataFrame.map`, `cumsum`, `diff`, `rolling` basics, and writing efficient custom transformations.